In [2]:
import os, cv2, numpy as np, torch, random, pandas as pd, matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from PIL import Image

# --- Basic Config ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Using device: {device}")

# --- Create base working directories ---
base_dir = "/kaggle/working/exposure_project"
os.makedirs(base_dir, exist_ok=True)

print("✅ Environment setup complete.")

 Using device: cuda
✅ Environment setup complete.


In [ ]:
adobe_path='/kaggle/input/adobe-fivek' 
lvzhdr_path='/kaggle/input/lvzhdr-tone-mapping-benchmark-dataset-tmonet' 
print("Done")

Done


In [ ]:
def collect_images(base_path, valid_exts=(".jpg", ".jpeg", ".png", ".hdr", ".exr")):
    images = []
    for root, _, files in os.walk(base_path):
        for f in files:
            if f.lower().endswith(valid_exts):
                images.append(os.path.join(root, f))
    return images

adobe_images = collect_images(adobe_path)
lvz_images = collect_images(lvzhdr_path)

print(f"Found {len(adobe_images)} Adobe images, {len(lvz_images)} LVZ-HDR images")

adobe_sample = random.sample(adobe_images, min(len(adobe_images), 2400))
lvz_sample = random.sample(lvz_images, min(len(lvz_images), 1600))

combined_images = adobe_sample + lvz_sample
random.shuffle(combined_images)

# --- Create DataFrame ---
df = pd.DataFrame(combined_images, columns=["image_path"])
df["dataset"] = ["adobe" if "fivek" in p.lower() else "lvzhdr" for p in df["image_path"]]

# --- Split into Train / Val / Test (80 / 10 / 10) ---
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, shuffle=True)

# --- Summary ---
print("\n📊 Dataset Split Summary:")
print(f"Train: {len(train_df)}  ({len(train_df)/len(df):.1%})")
print(f"Val:   {len(val_df)}  ({len(val_df)/len(df):.1%})")
print(f"Test:  {len(test_df)}  ({len(test_df)/len(df):.1%})")

# --- Save CSVs for reference ---
train_df.to_csv(os.path.join(base_dir, "train_split.csv"), index=False)
val_df.to_csv(os.path.join(base_dir, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(base_dir, "test_split.csv"), index=False)

print("\n✅ Dataset prepared and saved successfully.")

Found 30000 Adobe images, 457 LVZ-HDR images

📊 Dataset Split Summary:
Train: 2285  (80.0%)
Val:   286  (10.0%)
Test:  286  (10.0%)

✅ Dataset prepared and saved successfully.


## Getting Pairs

In [5]:
import os, cv2, numpy as np
import pandas as pd
from tqdm import tqdm

base_dir = "/kaggle/working/exposure_project"   # where your split CSVs are
train_split_csv = os.path.join(base_dir, "train_split.csv")
val_split_csv   = os.path.join(base_dir, "val_split.csv")
test_split_csv  = os.path.join(base_dir, "test_split.csv")

# 1) ROOT FOLDERS – ADJUST THESE TO YOUR SETUP
adobe_root = "/kaggle/input/adobe-fivek"   # MIT-Adobe 5K Kaggle root
adobe_raw_folder = "raw"                   # folder name for raw/low-quality
adobe_expert_folder = "e"                  # choose expert (a/b/c/d/e)

lvz_root = "/kaggle/input/lvzhdr-tone-mapping-benchmark-dataset-tmonet"
lvz_hdr_folder = "LVZ-HDR Dataset"         # folder (or subpath) with HDR
lvz_ldr_folder = "TMO-Net-Results on LVZ" # folder (or subpath) with LDR

# ---------- Helper functions ----------

def mean_luminance(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    return float(gray.mean())

def compute_ev(input_path, target_path, eps=1e-6):
    m_in = mean_luminance(input_path)
    m_tg = mean_luminance(target_path)
    if m_in is None or m_tg is None:
        return np.nan
    ratio = (m_tg + eps) / (m_in + eps)
    # EV ≈ log2(brightness_ratio)
    return float(np.log2(ratio))

def adobe_pair_paths(image_path):
    """
    Convert an Adobe 'image_path' to (input_path, target_path)
    assuming filenames match between raw/ and expert folders.
    Works whether current path points to raw/ or expert folder.
    """
    # Get only the filename part
    fname = os.path.basename(image_path)

    raw_path    = os.path.join(adobe_root, adobe_raw_folder, fname)
    expert_path = os.path.join(adobe_root, adobe_expert_folder, fname)

    # Decide which is more likely valid as input; prioritise raw as input
    if os.path.exists(raw_path) and os.path.exists(expert_path):
        input_path, target_path = raw_path, expert_path
    elif os.path.exists(raw_path):
        # Only raw exists -> cannot form pair
        return None, None
    elif os.path.exists(expert_path):
        # Only expert exists -> cannot form pair
        return None, None
    else:
        return None, None

    return input_path, target_path

def lvz_pair_paths(image_path):
    """
    Convert an LVZ-HDR 'image_path' to (input_path, target_path)
    assuming matching names between HDR and LDR result folders.
    You may need to tweak folder names depending on your layout.
    """
    fname = os.path.basename(image_path)
    # change extension if necessary (e.g., HDR .exr vs LDR .png)
    name_no_ext, _ = os.path.splitext(fname)
    hdr_path = os.path.join(lvz_root, lvz_hdr_folder, fname)
    # try common LDR extensions
    ldr_candidates = [
        os.path.join(lvz_root, lvz_ldr_folder, name_no_ext + ".png"),
        os.path.join(lvz_root, lvz_ldr_folder, name_no_ext + ".jpg"),
        os.path.join(lvz_root, lvz_ldr_folder, name_no_ext + ".jpeg"),
    ]
    ldr_path = next((p for p in ldr_candidates if os.path.exists(p)), None)

    if os.path.exists(hdr_path) and ldr_path is not None:
        return hdr_path, ldr_path
    else:
        return None, None

def build_pairs(split_df):
    rows = []
    for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
        image_path = row["image_path"]
        dataset = row["dataset"]

        if dataset == "adobe":
            inp, tgt = adobe_pair_paths(image_path)
        else:
            inp, tgt = lvz_pair_paths(image_path)

        if inp is None or tgt is None:
            # skip if pair not found
            continue

        ev = compute_ev(inp, tgt)
        rows.append({
            "input_path": inp,
            "target_path": tgt,
            "dataset": dataset,
            "ev_label": ev
        })

    return pd.DataFrame(rows)

# ---------- Build paired CSVs ----------

train_split = pd.read_csv(train_split_csv)
val_split   = pd.read_csv(val_split_csv)
test_split  = pd.read_csv(test_split_csv)

train_pairs = build_pairs(train_split)
val_pairs   = build_pairs(val_split)
test_pairs  = build_pairs(test_split)

print("Train pairs:", len(train_pairs))
print("Val pairs:", len(val_pairs))
print("Test pairs:", len(test_pairs))

train_pairs.to_csv(os.path.join(base_dir, "train_pairs.csv"), index=False)
val_pairs.to_csv(os.path.join(base_dir, "val_pairs.csv"), index=False)
test_pairs.to_csv(os.path.join(base_dir, "test_pairs.csv"), index=False)

print("✅ Saved train_pairs.csv, val_pairs.csv, test_pairs.csv")


100%|██████████| 286/286 [01:28<00:00,  3.22it/s]

Train pairs: 1911
Val pairs: 251
Test pairs: 238
✅ Saved train_pairs.csv, val_pairs.csv, test_pairs.csv


In [6]:
import cv2, torch, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader

class ToneMappingDataset(Dataset):
    def __init__(self, csv_path, hist_bins=64, img_size=(224, 224)):
        self.df = pd.read_csv(csv_path)
        self.hist_bins = hist_bins
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def _load_rgb(self, path):
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"Failed to read {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, self.img_size)
        img = img.astype(np.float32) / 255.0
        return img

    def _compute_hist(self, img_np):
        # img_np: HxWx3, [0,1]
        gray = cv2.cvtColor((img_np * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        hist = cv2.calcHist([gray], [0], None, [self.hist_bins], [0, 256]).flatten()
        hist = hist / (hist.sum() + 1e-8)
        return hist.astype(np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        inp = self._load_rgb(row["input_path"])
        tgt = self._load_rgb(row["target_path"])

        # to tensors
        inp_t = torch.from_numpy(inp).permute(2, 0, 1)  # [3,H,W]
        tgt_t = torch.from_numpy(tgt).permute(2, 0, 1)

        hist_np = self._compute_hist(inp)
        hist_t = torch.from_numpy(hist_np)             # [hist_bins]

        ev = torch.tensor([row["ev_label"]], dtype=torch.float32)  # [1]

        return inp_t, hist_t, ev, tgt_t

In [7]:
base_dir = "/kaggle/working/exposure_project"

train_ds = ToneMappingDataset(os.path.join(base_dir, "train_pairs.csv"))
val_ds   = ToneMappingDataset(os.path.join(base_dir, "val_pairs.csv"))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

# quick sanity check
batch = next(iter(train_loader))
inp, hist, ev, tgt = batch
print("Input:", inp.shape)   # [B,3,224,224]
print("Hist:", hist.shape)   # [B,64]
print("EV:", ev.shape)       # [B,1]
print("Target:", tgt.shape)  # [B,3,224,224]


Input: torch.Size([8, 3, 224, 224])
Hist: torch.Size([8, 64])
EV: torch.Size([8, 1])
Target: torch.Size([8, 3, 224, 224])


(encoder + histogram fusion + EV/scene heads + reconstruction net

In [8]:
import torch
import torch.nn as nn
import torchvision.models as models

class FusionEVModule(nn.Module):
    def __init__(self, hist_bins=64, backbone_name="resnet18", ev_out_dim=3, num_scenes=4):
        super().__init__()
        base = getattr(models, backbone_name)(weights=None)
        self.encoder = nn.Sequential(*list(base.children())[:-1])  # remove final fc
        enc_dim = base.fc.in_features

        self.hist_branch = nn.Sequential(
            nn.Linear(hist_bins, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
        )
        hist_dim = 64

        fused_dim = enc_dim + hist_dim
        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

        self.ev_head = nn.Linear(256, ev_out_dim)      # regression: outputs K values [B,K]
        self.scene_head = nn.Linear(256, num_scenes)   # optional scene classes

    def forward(self, x, hist):
        feat = self.encoder(x)              # [B,enc_dim,1,1]
        feat = feat.squeeze(-1).squeeze(-1)
        h = self.hist_branch(hist)         # [B,hist_dim]
        z = torch.cat([feat, h], dim=1)
        z = self.fusion(z)
        ev_pred = self.ev_head(z)          # [B,K] where K=ev_out_dim
        scene_logits = self.scene_head(z)  # [B,num_scenes]
        return ev_pred, scene_logits

In [9]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ReconUNet(nn.Module):
    def __init__(self, in_ch=4, out_ch=3):
        super().__init__()
        self.down1 = DoubleConv(in_ch, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)
        self.down4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv4 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv1 = DoubleConv(64, 32)

        self.out_conv = nn.Conv2d(32, out_ch, 1)

    def forward(self, x):
        d1 = self.down1(x); p1 = self.pool1(d1)
        d2 = self.down2(p1); p2 = self.pool2(d2)
        d3 = self.down3(p2); p3 = self.pool3(d3)
        d4 = self.down4(p3); p4 = self.pool4(d4)

        bn = self.bottleneck(p4)

        u4 = self.up4(bn); u4 = torch.cat([u4, d4], dim=1); u4 = self.conv4(u4)
        u3 = self.up3(u4); u3 = torch.cat([u3, d3], dim=1); u3 = self.conv3(u3)
        u2 = self.up2(u3); u2 = torch.cat([u2, d2], dim=1); u2 = self.conv2(u2)
        u1 = self.up1(u2); u1 = torch.cat([u1, d1], dim=1); u1 = self.conv1(u1)

        out = torch.sigmoid(self.out_conv(u1))
        return out


In [10]:
class FullToneMapModel(nn.Module):
    def __init__(self, hist_bins=64):
        super().__init__()
        self.ev_module = FusionEVModule(hist_bins=hist_bins, ev_out_dim=1)
        self.recon_net = ReconUNet(in_ch=4, out_ch=3)

    def forward(self, x, hist):
        ev_pred, scene_logits = self.ev_module(x, hist)
        # broadcast EV to image size
        B, _, H, W = x.shape
        ev_map = ev_pred.view(B, 1, 1, 1).expand(B, 1, H, W)
        recon_in = torch.cat([x, ev_map], dim=1)   # [B,4,H,W]
        out_img = self.recon_net(recon_in)
        return out_img, ev_pred, scene_logits

In [11]:
import os
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import cv2
import numpy as np

base_dir = "/kaggle/working/exposure_project"
os.makedirs(base_dir, exist_ok=True)

# Datasets / loaders (reuse existing class)
train_ds = ToneMappingDataset(os.path.join(base_dir, "train_pairs.csv"))
val_ds   = ToneMappingDataset(os.path.join(base_dir, "val_pairs.csv"))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = FullToneMapModel(hist_bins=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 5

def save_samples(inp, out_img, tgt, epoch, max_save=4):
    """Save a few input / output / target triplets for visual check."""
    inp = inp.cpu().numpy()
    out_img = out_img.cpu().detach().numpy()
    tgt = tgt.cpu().numpy()

    B = min(inp.shape[0], max_save)
    for i in range(B):
        in_im  = np.transpose(inp[i], (1, 2, 0))
        out_im = np.transpose(out_img[i], (1, 2, 0))
        tgt_im = np.transpose(tgt[i], (1, 2, 0))

        for name, im in [("input", in_im), ("pred", out_im), ("target", tgt_im)]:
            im_clipped = np.clip(im * 255.0, 0, 255).astype(np.uint8)
            im_bgr = cv2.cvtColor(im_clipped, cv2.COLOR_RGB2BGR)
            save_path = os.path.join(
                base_dir, f"samples_epoch{epoch}_{i}_{name}.png"
            )
            cv2.imwrite(save_path, im_bgr)

for epoch in range(1, num_epochs + 1):
    # ---- Train ----
    model.train()
    train_loss = 0.0
    for inp, hist, ev, tgt in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
        inp, hist, ev, tgt = inp.to(device), hist.to(device), ev.to(device), tgt.to(device)

        optimizer.zero_grad()
        out_img, ev_pred, scene_logits = model(inp, hist)

        l_recon = F.l1_loss(out_img, tgt)
        l_ev = F.mse_loss(ev_pred, ev)
        loss = l_recon + 0.1 * l_ev

        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inp.size(0)

    train_loss /= len(train_ds)

    # ---- Validation ----
    model.eval()
    val_loss = 0.0
    first_batch_saved = False
    with torch.no_grad():
        for inp, hist, ev, tgt in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
            inp, hist, ev, tgt = inp.to(device), hist.to(device), ev.to(device), tgt.to(device)
            out_img, ev_pred, scene_logits = model(inp, hist)

            l_recon = F.l1_loss(out_img, tgt)
            l_ev = F.mse_loss(ev_pred, ev)
            loss = l_recon + 0.1 * l_ev
            val_loss += loss.item() * inp.size(0)

            # save samples from first val batch
            if not first_batch_saved:
                save_samples(inp, out_img, tgt, epoch)
                first_batch_saved = True

    val_loss /= len(val_ds)

    print(f"Epoch {epoch}: train loss = {train_loss:.4f} | val loss = {val_loss:.4f}")

Epoch 1 [train]:   0%|          | 0/239 [00:00<?, ?it/s]

Epoch 1 [val]:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 1: train loss = 0.1423 | val loss = 0.1144


Epoch 2 [train]:   0%|          | 0/239 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f4b4ed09260>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7f4b4ed09260>
^Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
^    self._shutdown_workers()
^  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
^    ^^if w.is_alive():^^^
^ ^ ^ ^ ^ ^  ^^^^^^^

...

Epoch 4 [val]:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 4: train loss = 0.0931 | val loss = 0.0966


Epoch 5 [train]:   0%|          | 0/239 [00:00<?, ?it/s]

Epoch 5 [val]:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 5: train loss = 0.0890 | val loss = 0.0855


In [12]:
from dataclasses import dataclass

@dataclass
class TrainConfig:
    epochs: int = 30
    batch_size: int = 16
    lr: float = 1e-4
    weight_decay: float = 1e-5
    ev_loss_weight: float = 0.1
    grad_clip: float = 1.0
    hist_bins: int = 64
    img_size: tuple = (224, 224)
    backbone: str = "resnet18"
    num_workers: int = 4
    save_every: int = 5            # epochs
    sample_every: int = 1          # epochs
    ckpt_dir: str = "/kaggle/working/exposure_project/checkpoints"
    sample_dir: str = "/kaggle/working/exposure_project/samples"

In [13]:
import os, torch
from torch.utils.data import DataLoader

def make_loaders(cfg: TrainConfig, base_dir="/kaggle/working/exposure_project"):
    train_ds = ToneMappingDataset(os.path.join(base_dir, "train_pairs.csv"),
                                  hist_bins=cfg.hist_bins, img_size=cfg.img_size)
    val_ds   = ToneMappingDataset(os.path.join(base_dir, "val_pairs.csv"),
                                  hist_bins=cfg.hist_bins, img_size=cfg.img_size)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size,
                              shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    return train_loader, val_loader, train_ds, val_ds

def make_model(cfg: TrainConfig, device):
    model = FullToneMapModel(hist_bins=cfg.hist_bins).to(device)
    return model

In [14]:
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm
import numpy as np
import cv2

def save_samples(inp, out_img, tgt, epoch, cfg: TrainConfig):
    os.makedirs(cfg.sample_dir, exist_ok=True)
    inp = inp.cpu().numpy()
    out_img = out_img.cpu().detach().numpy()
    tgt = tgt.cpu().numpy()
    B = min(inp.shape[0], 4)
    for i in range(B):
        for name, im in [("input", inp[i]), ("pred", out_img[i]), ("target", tgt[i])]:
            im_hw = np.transpose(im, (1, 2, 0))
            im_clipped = np.clip(im_hw * 255.0, 0, 255).astype(np.uint8)
            im_bgr = cv2.cvtColor(im_clipped, cv2.COLOR_RGB2BGR)
            path = os.path.join(cfg.sample_dir, f"epoch{epoch:03d}_{i}_{name}.png")
            cv2.imwrite(path, im_bgr)

def save_checkpoint(model, optimizer, epoch, best_val, cfg: TrainConfig):
    os.makedirs(cfg.ckpt_dir, exist_ok=True)
    path = os.path.join(cfg.ckpt_dir, f"model_epoch{epoch:03d}.pth")
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val": best_val,
    }, path)

def train_loop(cfg: TrainConfig):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    train_loader, val_loader, train_ds, val_ds = make_loaders(cfg)
    model = make_model(cfg, device)

    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr,
                            weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3, verbose=True
    )

    best_val = float("inf")

    for epoch in range(1, cfg.epochs + 1):
        # ---- Train ----
        model.train()
        train_loss = 0.0
        for inp, hist, ev, tgt in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
            inp, hist, ev, tgt = inp.to(device), hist.to(device), ev.to(device), tgt.to(device)

            optimizer.zero_grad(set_to_none=True)
            out_img, ev_pred, scene_logits = model(inp, hist)

            l_recon = F.l1_loss(out_img, tgt)
            l_ev = F.mse_loss(ev_pred, ev)
            loss = l_recon + cfg.ev_loss_weight * l_ev

            loss.backward()
            if cfg.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()

            train_loss += loss.item() * inp.size(0)

        train_loss /= len(train_ds)

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            first_batch = True
            for inp, hist, ev, tgt in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
                inp, hist, ev, tgt = inp.to(device), hist.to(device), ev.to(device), tgt.to(device)
                out_img, ev_pred, scene_logits = model(inp, hist)

                l_recon = F.l1_loss(out_img, tgt)
                l_ev = F.mse_loss(ev_pred, ev)
                loss = l_recon + cfg.ev_loss_weight * l_ev
                val_loss += loss.item() * inp.size(0)

                if first_batch and (epoch % cfg.sample_every == 0):
                    save_samples(inp, out_img, tgt, epoch, cfg)
                    first_batch = False

        val_loss /= len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch}: train={train_loss:.4f} | val={val_loss:.4f} | lr={optimizer.param_groups[0]['lr']:.2e}")

        # checkpointing
        if val_loss < best_val:
            best_val = val_loss
            save_checkpoint(model, optimizer, epoch, best_val, cfg)
        elif epoch % cfg.save_every == 0:
            save_checkpoint(model, optimizer, epoch, best_val, cfg)

    print(f"Best val loss: {best_val:.4f}")
    return model


In [15]:
cfg = TrainConfig(
    epochs=20,
    batch_size=16,
    lr=1e-4,
    ev_loss_weight=0.1,
    img_size=(224, 224),
    backbone="resnet18"
)

model = train_loop(cfg)


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 [train]:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch 1 [val]:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 1: train=0.1463 | val=0.1151 | lr=1.00e-04


Epoch 2 [train]:   0%|          | 0/120 [00:00<?, ?it/s]

...

Epoch 19 [val]:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 19: train=0.0594 | val=0.0725 | lr=5.00e-05


Epoch 20 [train]:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch 20 [val]:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 20: train=0.0586 | val=0.0716 | lr=5.00e-05
Best val loss: 0.0709


In [16]:
import os, torch

from torch.utils.data import DataLoader

cfg = TrainConfig()  # use same settings as training
device = "cuda" if torch.cuda.is_available() else "cpu"

# find best checkpoint manually (set epoch)
best_ckpt_path = "/kaggle/working/exposure_project/checkpoints/model_epoch020.pth"

ckpt = torch.load(best_ckpt_path, map_location=device)
model = FullToneMapModel(hist_bins=cfg.hist_bins).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()

print("Loaded checkpoint from epoch:", ckpt["epoch"], "best_val:", ckpt["best_val"])


Loaded checkpoint from epoch: 20 best_val: 0.07093986628421274


In [17]:
import torch.nn.functional as F
import numpy as np
from math import log10

# simple SSIM implementation (patch-based, fast enough for eval)
def ssim_torch(x, y, C1=0.01**2, C2=0.03**2):
    # x, y: [B,3,H,W] in [0,1]
    mu_x = F.avg_pool2d(x, 3, 1, 1)
    mu_y = F.avg_pool2d(y, 3, 1, 1)
    sigma_x = F.avg_pool2d(x * x, 3, 1, 1) - mu_x ** 2
    sigma_y = F.avg_pool2d(y * y, 3, 1, 1) - mu_y ** 2
    sigma_xy = F.avg_pool2d(x * y, 3, 1, 1) - mu_x * mu_y

    ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / (
        (mu_x**2 + mu_y**2 + C1) * (sigma_x + sigma_y + C2)
    )
    return ssim_map.mean()

base_dir = "/kaggle/working/exposure_project"
test_ds = ToneMappingDataset(os.path.join(base_dir, "test_pairs.csv"),
                             hist_bins=cfg.hist_bins, img_size=cfg.img_size)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=0)

total_l1, total_mse, total_psnr, total_ssim, n = 0.0, 0.0, 0.0, 0.0, 0

with torch.no_grad():
    for inp, hist, ev, tgt in test_loader:
        inp, hist, ev, tgt = inp.to(device), hist.to(device), ev.to(device), tgt.to(device)
        out_img, ev_pred, scene_logits = model(inp, hist)

        # L1 & MSE
        l1 = F.l1_loss(out_img, tgt, reduction="sum")
        mse = F.mse_loss(out_img, tgt, reduction="sum")

        # PSNR (peak=1 since images in [0,1])
        mse_batch = mse.item() / (out_img.numel())
        psnr_batch = 10 * log10(1.0 / (mse_batch + 1e-8))

        # SSIM
        ssim_batch = ssim_torch(out_img, tgt).item()

        bsz = inp.size(0)
        total_l1 += l1.item()
        total_mse += mse.item()
        total_psnr += psnr_batch * bsz
        total_ssim += ssim_batch * bsz
        n += bsz

print(f"Test L1  : {total_l1 / n:.4f}")
print(f"Test MSE : {total_mse / n:.4f}")
print(f"Test PSNR: {total_psnr / n:.2f} dB")
print(f"Test SSIM: {total_ssim / n:.4f}")


Test L1  : 10461.3045
Test MSE : 1474.0296
Test PSNR: 20.31 dB
Test SSIM: 0.6852


In [18]:
import random, cv2, numpy as np

sample_dir = os.path.join(base_dir, "test_samples")
os.makedirs(sample_dir, exist_ok=True)

def save_triplet(inp_t, out_t, tgt_t, idx):
    inp = inp_t.cpu().numpy()
    out_img = out_t.cpu().numpy()
    tgt = tgt_t.cpu().numpy()

    for name, im in [("input", inp), ("pred", out_img), ("target", tgt)]:
        im_hw = np.transpose(im, (1, 2, 0))
        im_clipped = np.clip(im_hw * 255.0, 0, 255).astype(np.uint8)
        im_bgr = cv2.cvtColor(im_clipped, cv2.COLOR_RGB2BGR)
        path = os.path.join(sample_dir, f"sample_{idx}_{name}.png")
        cv2.imwrite(path, im_bgr)

# pick 10 random indices
indices = random.sample(range(len(test_ds)), min(10, len(test_ds)))

with torch.no_grad():
    for i, idx in enumerate(indices):
        inp, hist, ev, tgt = test_ds[idx]
        inp_b = inp.unsqueeze(0).to(device)
        hist_b = hist.unsqueeze(0).to(device)
        out_img, _, _ = model(inp_b, hist_b)
        save_triplet(inp, out_img[0].cpu(), tgt, i)

print("Saved visual test samples to:", sample_dir)


Saved visual test samples to: /kaggle/working/exposure_project/test_samples


In [19]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class ReconUNetV2(nn.Module):
    def __init__(self, in_ch=9, out_ch=3):
        super().__init__()
        self.down1 = DoubleConv(in_ch, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)
        self.down4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv4 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv1 = DoubleConv(64, 32)

        self.out_conv = nn.Conv2d(32, out_ch, 1)

    def forward(self, x):
        d1 = self.down1(x); p1 = self.pool1(d1)
        d2 = self.down2(p1); p2 = self.pool2(d2)
        d3 = self.down3(p2); p3 = self.pool3(d3)
        d4 = self.down4(p3); p4 = self.pool4(d4)

        bn = self.bottleneck(p4)

        u4 = self.up4(bn)
        u4 = torch.cat([u4, d4], dim=1)
        u4 = self.conv4(u4)

        u3 = self.up3(u4)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.conv3(u3)

        u2 = self.up2(u3)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.conv2(u2)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.conv1(u1)

        out = torch.sigmoid(self.out_conv(u1))
        return out


# Exposure Boosting Network Architecture

This revised architecture implements the **Exposure Boosting Strategy** for robust tone mapping. 

### Methodology
Unlike traditional methods that condition on a single scalar EV, our model mimics a **multi-exposure bracketing** workflow within the network:

1.  **Multi-EV Prediction**: The `FusionEVModule` predicts 3 distinct exposure values: **Underexposed**, **Normal**, and **Overexposed**.
2.  **Virtual Exposure Generation**: Using the input raw/sRGB image $I$, we generate 3 virtual exposures using a differentiable gain function:
    $$ I_{k} = \text{clamp}(I \times 2^{EV_k}, 0, 1) $$
3.  **Deep Fusion via Stacking**: These 3 images are concatenated channel-wise to form a **9-channel tensor** ($3 \times 3$ RGB channels).
4.  **Boosted Reconstruction**: The `ReconUNetV2` takes this rich 9-channel input to produce the final high-quality tone-mapped image. This allows the network to effectively 'see' shadow details from the overexposed virtual view and highlight details from the underexposed view simultaneously.


In [20]:
class FullToneMapModelBoosting(nn.Module):
    def __init__(self, hist_bins=64, ev_out_dim=3):
        super().__init__()
        self.ev_out_dim = ev_out_dim
        self.ev_module = FusionEVModule(hist_bins=hist_bins, ev_out_dim=ev_out_dim)
        # Input channels: 3 images * 3 channels each = 9 channels
        self.recon_net = ReconUNetV2(in_ch=9, out_ch=3)

    def forward(self, x, hist):
        # 1. Predict EVs
        ev_pred, scene_logits = self.ev_module(x, hist) # [B, 3]
        
        # 2. Virtual Exposure Generation
        # I_ev = clamp(I * 2^EV, 0, 1)
        B, _, H, W = x.shape
        exposure_imgs = []
        
        for k in range(self.ev_out_dim):
            ev_k = ev_pred[:, k].view(B, 1, 1, 1)
            # Differentiable exposure gain
            gain = torch.pow(2.0, ev_k)
            img_k = x * gain
            img_k = torch.clamp(img_k, 0.0, 1.0)
            exposure_imgs.append(img_k)
            
        # 3. Stack virtual exposures channel-wise
        # Result: [B, 9, H, W]
        recon_in = torch.cat(exposure_imgs, dim=1)
        
        # 4. Reconstruct with fused information
        out_img = self.recon_net(recon_in)
        
        return out_img, ev_pred, scene_logits, exposure_imgs


In [21]:
from math import log10

# SSIM
def ssim_torch(x, y, C1=0.01**2, C2=0.03**2):
    mu_x = F.avg_pool2d(x, 3, 1, 1)
    mu_y = F.avg_pool2d(y, 3, 1, 1)
    sigma_x = F.avg_pool2d(x * x, 3, 1, 1) - mu_x ** 2
    sigma_y = F.avg_pool2d(y * y, 3, 1, 1) - mu_y ** 2
    sigma_xy = F.avg_pool2d(x * y, 3, 1, 1) - mu_x * mu_y

    ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / (
        (mu_x**2 + mu_y**2 + C1) * (sigma_x + sigma_y + C2)
    )
    return ssim_map.mean()

# VGG perceptual features
device = "cuda" if torch.cuda.is_available() else "cpu"
vgg_feat = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_FEATURES).features[:16].to(device)
for p in vgg_feat.parameters():
    p.requires_grad = False

def perceptual_loss(x, y):
    x_n = (x - 0.5) * 2
    y_n = (y - 0.5) * 2
    fx = vgg_feat(x_n)
    fy = vgg_feat(y_n)
    return F.l1_loss(fx, fy)

def total_loss_v2(out_img, tgt, ev_pred, ev_gt, ev_weight=0.03):
    l1 = F.l1_loss(out_img, tgt)
    ssim_loss = 1 - ssim_torch(out_img, tgt)
    lp = perceptual_loss(out_img, tgt)
    lev = F.mse_loss(ev_pred, ev_gt)
    return l1 + 0.2 * ssim_loss + 0.05 * lp + ev_weight * lev


Downloading: "https://download.pytorch.org/models/vgg16_features-amdegroot-88682ab5.pth" to /root/.cache/torch/hub/checkpoints/vgg16_features-amdegroot-88682ab5.pth
100%|██████████| 528M/528M [00:02<00:00, 190MB/s] 


In [22]:
from dataclasses import dataclass
from torch.utils.data import DataLoader

@dataclass
class TrainConfigV2:
    epochs: int = 40
    batch_size: int = 16
    lr: float = 1e-4
    weight_decay: float = 1e-5
    grad_clip: float = 1.0
    hist_bins: int = 64
    img_size: tuple = (224, 224)
    num_workers: int = 2
    ev_out_dim: int = 3  # 3 EVs: Under, Mid, Over
    ev_center_weight: float = 0.03
    ev_order_weight: float = 0.01
    ev_space_weight: float = 0.01
    ev_spacing_delta: float = 0.7
    ckpt_dir: str = "/kaggle/working/exposure_project/v2_checkpoints"
    sample_dir: str = "/kaggle/working/exposure_project/v2_samples"

def make_loaders_v2(cfg: TrainConfigV2, base_dir="/kaggle/working/exposure_project"):
    train_ds = ToneMappingDataset(os.path.join(base_dir, "train_pairs.csv"),
                                  hist_bins=cfg.hist_bins, img_size=cfg.img_size)
    val_ds   = ToneMappingDataset(os.path.join(base_dir, "val_pairs.csv"),
                                  hist_bins=cfg.hist_bins, img_size=cfg.img_size)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              shuffle=True, num_workers=cfg.num_workers,
                              pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size,
                              shuffle=False, num_workers=cfg.num_workers,
                              pin_memory=True, persistent_workers=True)
    return train_loader, val_loader, train_ds, val_ds


In [23]:
# --- SAFETY & DEBUG CHECK ---
# Run a dummy forward pass to verify shapes and causality before full training.

if torch.cuda.is_available():
    print("\u2705 Running Safety Check...")
    debug_device = "cuda"
    debug_model = FullToneMapModelBoosting(hist_bins=64, ev_out_dim=3).to(debug_device)
    
    # Dummy Data: [B=2, C=3, H=224, W=224]
    debug_in = torch.randn(2, 3, 224, 224).to(debug_device)
    debug_hist = torch.randn(2, 64).to(debug_device)
    
    with torch.no_grad():
        d_out, d_ev, _, d_imgs = debug_model(debug_in, debug_hist)
        
    print(f"Out Image Shape: {d_out.shape} (Expected [2, 3, 224, 224])")
    print(f"EV Prediction Shape: {d_ev.shape} (Expected [2, 3])")
    print(f"Virtual Exposures: {len(d_imgs)} images (Expected 3)")
    if len(d_imgs) > 0:
        print(f"Virtual Img[0] Shape: {d_imgs[0].shape}")
        
    # Check Stacking Logic implicitly by success of forward pass
    print(f"Sample EVs (Batch 0): {d_ev[0].tolist()}")
    
    # Simple Causality / Ordering Check
    # We want EV_under < EV_mid < EV_over generally, though not guaranteed at init.
    # Just printing them is enough for the user to see.
    print("\u2705 Safety Check Complete. Shapes look correct.")


✅ Running Safety Check...
Out Image Shape: torch.Size([2, 3, 224, 224]) (Expected [2, 3, 224, 224])
EV Prediction Shape: torch.Size([2, 3]) (Expected [2, 3])
Virtual Exposures: 3 images (Expected 3)
Virtual Img[0] Shape: torch.Size([2, 3, 224, 224])
Sample EVs (Batch 0): [0.09086108952760696, -0.1223268061876297, -0.010029658675193787]
✅ Safety Check Complete. Shapes look correct.


In [24]:
import os, csv
import torch.optim as optim
import torch.nn.functional as F
from tqdm.auto import tqdm
import numpy as np, cv2

def save_samples_v2(inp, out_img, tgt, epoch, cfg: TrainConfigV2):
    os.makedirs(cfg.sample_dir, exist_ok=True)
    inp = inp.cpu().numpy()
    out_img = out_img.cpu().detach().numpy()
    tgt = tgt.cpu().numpy()
    B = min(inp.shape[0], 4)
    for i in range(B):
        for name, im in [("input", inp[i]), ("pred", out_img[i]), ("target", tgt[i])]:
            im_hw = np.transpose(im, (1, 2, 0))
            im_clipped = np.clip(im_hw * 255.0, 0, 255).astype(np.uint8)
            im_bgr = cv2.cvtColor(im_clipped, cv2.COLOR_RGB2BGR)
            path = os.path.join(cfg.sample_dir, f"epoch{epoch:03d}_{i}_{name}.png")
            cv2.imwrite(path, im_bgr)

def save_ev_bracket_previews(exposure_imgs, epoch, cfg: TrainConfigV2):
    # exposure_imgs is a list of [B, 3, H, W] tensors
    os.makedirs(cfg.sample_dir, exist_ok=True)
    sample_idx = 0
    names = ['under', 'mid', 'over']
    
    if len(exposure_imgs) > 0 and exposure_imgs[0].size(0) > sample_idx:
         # Just grab the first sample from the batch
         for k, img_tensor in enumerate(exposure_imgs):
             img = img_tensor[sample_idx].cpu().detach().numpy()
             img_hw = np.transpose(img, (1, 2, 0))
             img_clipped = np.clip(img_hw * 255.0, 0, 255).astype(np.uint8)
             img_bgr = cv2.cvtColor(img_clipped, cv2.COLOR_RGB2BGR)
             
             # Try to map k to name if possible, else generic
             name = names[k] if k < len(names) else f'ev{k}'
             path = os.path.join(cfg.sample_dir, f"epoch{epoch:03d}_virtual_{name}.png")
             cv2.imwrite(path, img_bgr)

def save_ckpt_v2(model, optimizer, epoch, best_val, cfg: TrainConfigV2):
    os.makedirs(cfg.ckpt_dir, exist_ok=True)
    path = os.path.join(cfg.ckpt_dir, f"model_epoch{epoch:03d}.pth")
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val": best_val,
    }, path)

def compute_multi_ev_loss(ev_pred, ev_gt, cfg: TrainConfigV2):
    B, K = ev_pred.shape
    # Center EV (index 1) supervision
    mid = 1
    ev_center = ev_pred[:, mid:mid+1]
    l_center = F.mse_loss(ev_center, ev_gt)
    
    # Ordering: EV_under < EV_mid < EV_over
    l_order = 0.0
    for i in range(K - 1):
        diff = ev_pred[:, i] - ev_pred[:, i+1]
        l_order += F.relu(diff).mean()
    
    # Spacing: Ensure adequate separation
    l_space = 0.0
    for i in range(K - 1):
        gap = ev_pred[:, i+1] - ev_pred[:, i]
        l_space += F.relu(cfg.ev_spacing_delta - gap).mean()
    
    return l_center, l_order, l_space

def train_v2():
    cfg = TrainConfigV2()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Starting training on {device}...")

    train_loader, val_loader, train_ds, val_ds = make_loaders_v2(cfg)
    model = FullToneMapModelBoosting(hist_bins=cfg.hist_bins, ev_out_dim=cfg.ev_out_dim).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3, verbose=True
    )

    best_val = float("inf")

    for epoch in range(1, cfg.epochs + 1):
        # ---- train ----
        model.train()
        train_loss = 0.0
        for inp, hist, ev, tgt in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
            inp, hist, ev, tgt = inp.to(device), hist.to(device), ev.to(device), tgt.to(device)

            optimizer.zero_grad(set_to_none=True)
            out_img, ev_pred, scene_logits, _ = model(inp, hist)

            l_recon = F.l1_loss(out_img, tgt)
            ssim_loss = 1 - ssim_torch(out_img, tgt)
            lp = perceptual_loss(out_img, tgt)
            l_center, l_order, l_space = compute_multi_ev_loss(ev_pred, ev, cfg)
            
            loss = l_recon + 0.2 * ssim_loss + 0.05 * lp + \
                   cfg.ev_center_weight * l_center + \
                   cfg.ev_order_weight * l_order + \
                   cfg.ev_space_weight * l_space
            
            loss.backward()
            if cfg.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()

            train_loss += loss.item() * inp.size(0)

        train_loss /= len(train_ds)

        # ---- val ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            first_batch = True
            for inp, hist, ev, tgt in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
                inp, hist, ev, tgt = inp.to(device), hist.to(device), ev.to(device), tgt.to(device)
                out_img, ev_pred, scene_logits, exposure_imgs = model(inp, hist)
                
                l_recon = F.l1_loss(out_img, tgt)
                ssim_loss = 1 - ssim_torch(out_img, tgt)
                lp = perceptual_loss(out_img, tgt)
                l_center, l_order, l_space = compute_multi_ev_loss(ev_pred, ev, cfg)
                
                loss = l_recon + 0.2 * ssim_loss + 0.05 * lp + \
                       cfg.ev_center_weight * l_center + \
                       cfg.ev_order_weight * l_order + \
                       cfg.ev_space_weight * l_space
                val_loss += loss.item() * inp.size(0)

                if first_batch:
                    save_samples_v2(inp, out_img, tgt, epoch, cfg)
                    save_ev_bracket_previews(exposure_imgs, epoch, cfg)
                    first_batch = False

        val_loss /= len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch}: train={train_loss:.4f} | val={val_loss:.4f} | lr={optimizer.param_groups[0]['lr']:.2e}")

        os.makedirs(cfg.ckpt_dir, exist_ok=True)
        log_path = os.path.join(cfg.ckpt_dir, "train_log.csv")
        header_needed = not os.path.exists(log_path)
        with open(log_path, "a", newline="") as f:
            w = csv.writer(f)
            if header_needed:
                w.writerow(["epoch", "train_loss", "val_loss"])
            w.writerow([epoch, train_loss, val_loss])
        
        if val_loss < best_val:
            best_val = val_loss
            save_ckpt_v2(model, optimizer, epoch, best_val, cfg)
        elif epoch % 5 == 0:
            save_ckpt_v2(model, optimizer, epoch, best_val, cfg)

    print(f"Best val loss: {best_val:.4f}")
    return model

# Run:
model_v2 = train_v2()


Starting training on cuda...


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1 [train]:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch 1 [val]:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 1: train=0.3753 | val=0.2790 | lr=1.00e-04


...

Epoch 39 [val]:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 39: train=0.1619 | val=0.1780 | lr=2.50e-05


Epoch 40 [train]:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch 40 [val]:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 40: train=0.1611 | val=0.1791 | lr=2.50e-05
Best val loss: 0.1780


In [26]:
model_v2.eval()                     
inp, hist, ev_gt, tgt = next(iter(val_loader))
inp, hist = inp.to(device), hist.to(device)

with torch.no_grad():
    # Inference does NOT need special flags anymore
    out_img, ev_pred, _, exposure_imgs = model_v2(inp, hist)

print("Ground-truth EV (first 4):", ev_gt[:4].numpy().ravel())
print("Predicted EVs (first 4 samples): [Under, Mid, Over]")
print(ev_pred[:4].cpu().numpy())    


Ground-truth EV (first 4): [-0.27156088  0.65435684  0.43656817 -0.64651066]
Predicted EVs (first 4 samples): [Under, Mid, Over]
[[-1.5562469  -0.04815363  0.57255673]
 [-0.9986939   0.5637927   2.2738926 ]
 [-0.9257073   0.19482271  1.0358816 ]
 [-0.9391494  -0.15072545  0.43088636]]
